# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [4]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [6]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('AQ.') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(api_key=api_key, base_url=GEMINI_BASE_URL)

API key looks good so far


In [ ]:
links = fetch_website_links("https://edwarddonner.com")S
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [1]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [2]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [5]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'offerings page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'associated product/company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-2.5-flash
Found 8 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'affiliated company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 19 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'product offering', 'url': 'https://huggingface.co/models'},
  {'type': 'product offering', 'url': 'https://huggingface.co/datasets'},
  {'type': 'product offering', 'url': 'https://huggingface.co/spaces'},
  {'type': 'product offering', 'url': 'https://huggingface.co/storage'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise solutions', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'product offering', 'url': 'https://huggingface.co/chat'},
  {'type': 'company profiles', 'url': 'https://huggingface.co/organizations'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learning resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'github profile', 'url': 'https://github.com/huggingface'},
  {'type': 'product offering', 'url': 'https://huggingface.co/pro'},
  {'type':

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
9 days ago
•
1.68M
•
1.73k
zai-org/GLM-5.2
Updated
5 days ago
•
282k
•
3.57k
tencent/Hy3
Updated
1 day ago
•
121
•
448
baidu/Unlimited-OCR
Updated
5 days ago
•
1.08M
•
1.83k
InternScience/Agents-A1
Updated
4 days ago
•
14.7k
•
367
Brow

In [25]:
brochure_system_prompt = """
You are a Gen Z with plenty social media vocabulary assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 24 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n9 days ago\n•\n1.68M\n•\n1.73k\nzai-org/GLM-5.2\nUpdated\n5 days ago\n•

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 16 relevant links


Hugging Face: The AI Community Building the Future

Hugging Face is the premier platform empowering the machine learning (ML) community to create, discover, and collaborate on models, datasets, and applications. We are dedicated to accelerating the future of AI through an open, collaborative, and fast-paced environment.

**For Prospective Customers & ML Professionals:**
Hugging Face is your essential partner for machine learning development. Our platform provides:

*   **Vast Resources:** Access over 2 million diverse models, 500,000+ datasets, and 1 million+ interactive AI applications (Spaces) across various modalities including text and image.
*   **Seamless Collaboration:** Host and collaborate on unlimited public models, datasets, and applications, fostering a vibrant ecosystem of innovation.
*   **Accelerated Development:** Move faster with our robust open-source stack, comprehensive documentation, and tools like HuggingChat for conversational AI.
*   **Integrated Solutions:** Leverage Storage Buckets, Inference Endpoints, and a suite of community tools (Discord, Forum, GitHub) to streamline your ML workflow.

**For Enterprise Clients:**
Beyond our extensive community offerings, Hugging Face provides tailored solutions for businesses and larger organizations seeking to harness the power of AI:

*   **Hugging Face PRO:** Enhanced features and dedicated support for professional teams.
*   **Enterprise Support:** Critical assistance to ensure your mission-critical AI projects run smoothly.
*   **Scalable Inference:** Robust Inference Providers and Endpoints for efficient and reliable model deployment at scale.
*   **Secure Storage:** Dedicated Storage Buckets for managing your valuable AI assets.

**For Investors:**
Hugging Face stands at the forefront of the AI revolution, driving the global open-source AI ecosystem. We represent:

*   **Market Leadership:** Powering the collective intelligence of the machine learning community worldwide.
*   **Rapid Growth:** A dynamic and expanding team (186+ members and growing) continually innovating and expanding our platform's capabilities.
*   **Strategic Impact:** Our commitment to open source, research, and enterprise solutions positions us as a critical infrastructure provider in the evolving AI landscape.
*   **Strong Engagement:** A highly active community consistently contributing to papers, articles, and projects, demonstrating deep ecosystem health.

**For Recruits & Careers:**
Join a passionate and impactful team that is literally "building the future" of AI. At Hugging Face, you will find:

*   **A Culture of Openness:** We are deeply committed to open source, collaboration, and knowledge sharing. You'll contribute to a platform that empowers millions.
*   **Cutting-Edge Innovation:** Engage with the latest advancements in AI through daily papers, insightful articles, and hands-on development on our core open-source stack.
*   **Continuous Learning:** Grow your skills with resources like Hugging Face Fundamentals and thrive in an environment that values curiosity and exploration.
*   **Global Impact:** Work alongside a diverse and talented team, making a tangible difference in how AI is developed, shared, and utilized across the globe.

Hugging Face is more than just a platform; it's a movement towards a more collaborative, open, and accessible AI future.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 26 relevant links


✨ **Hugging Face: Your AI Main Character Moment!** ✨

Yo, fam! If you're into AI, then you already know the vibe. Hugging Face isn't just a platform; it's the ultimate collab space where the future of AI is getting built, no cap. We're talking next-level innovation, shared with the whole community.

---

### 🚀 **What We Do (The Deets):**

We're the home of machine learning, making it easy to create, discover, and collab like never before. Think of us as your go-to hub for all things AI, whether you're a solo dev or an enterprise slayin' the game.

*   **Models for Days:** Dive into over 2 MILLION models! From cutting-edge LLMs to specialized tools, you'll find what you need to make your projects ✨pop✨.
*   **Datasets Galore:** Got data? We've got 500k+ datasets ready for your training and fine-tuning needs. Less hunting, more building!
*   **Spaces & Apps that Slay:** Explore or launch over 1 MILLION AI applications. Real-time voice, powerful image editing, OCR – you name it, our community's probably built it. Get your app running and share it with the world!
*   **Open Source FTW:** We're all about that open-source life, pushing boundaries and moving faster together.

---

### 🤩 **Who's in Our Squad (Customers):**

*   **Innovators & Devs:** If you're an ML enthusiast, researcher, or dev tryna build the next big thing, this is your playground. Host your projects, share your wins, and connect with like-minded legends.
*   **Teams & Enterprises:** Scaling your AI ambitions? We got you. Our enterprise solutions offer dedicated support, secure inference, and robust storage to keep your business ahead of the curve. Get that professional edge with Hugging Face PRO!

---

### vibes ✨ **Our Culture (The Fam Feels):**

Hugging Face is all about **community, collaboration, and open-source everything.** We believe that building the future of AI shouldn't be a solo mission. It's about sharing, learning, and growing together. We're a global fam that thrives on innovation, curiosity, and making ML accessible to everyone. Our Discord, Forum, and GitHub are always buzzing – come join the convo and level up your skills!

---

### 👩‍💻 **Your Career Journey:**

While we don't have specific job listings here, just know that a company driving the "AI community building the future" is always looking for talent that brings main character energy! If you're passionate about ML, open source, and making an impact, keep an eye on our official channels. We're shaping tomorrow, and we need brilliant minds to make it happen.

---

Ready to dive in? Explore AI apps, browse millions of models, or just link up with the brightest minds in ML. Let's build the future, together!

**[Explore Now & Join the Community!](https://huggingface.co/)**

In [27]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 26 relevant links


✨ Hugging Face: Level Up Your AI Game! 🚀

Hey fam! Ever wondered where all the cool AI stuff happens? 👀 BTW, Hugging Face is *the* place where the whole machine learning squad unites to build, share, and vibe on the future of AI. Think of us as the ultimate collab platform for models, datasets, and applications. No cap!

**What We Do (For Real):**
We're basically bringing the entire ML ecosystem to your fingertips:
*   **Models:** Scroll through over 2 million (!!!) models, from cutting-edge text generation to sick image-to-video tools. Find what you need, or drop your own creations. It's giving main character energy for your projects.
*   **Datasets:** Got data? Need data? We've got 500k+ datasets ready for your next big thing.
*   **Spaces:** Flex your creativity with 1M+ interactive AI apps. Want a real-time voice chat or a powerful image editor? We got you.
*   **Enterprise Solutions:** For the big brains and big businesses, we offer next-level support, inference endpoints, and robust storage buckets. Basically, we help you scale your AI without the drama.

**Our Vibe & Why We're Different:**
We're all about that open-source life and making AI accessible for everyone. Our community is super active, always sharing the latest papers, dropping knowledge, and building together. We're moving fast, exploring everything from text to images, and keeping it collaborative. It's a whole mood. We're not just a platform; we're a movement.

**Who's Joining the Party?**
*   **Prospective Customers (Devs & Researchers):** If you're building, experimenting, or just curious, this is your playground to discover, create, and collaborate better.
*   **Investors:** See the massive potential in a thriving, open-source AI ecosystem with powerful enterprise offerings? We're shaping the future of ML.
*   **Recruits (Talent):** Wanna be part of a team literally shaping the future of AI, with a focus on open-source and a vibrant community? If you're passionate about ML and ready to move fast, slide into our DMs (aka, check out our team and enterprise solutions!).

**Ready to Make Moves?**
Tap in, explore the apps, browse models, or just link up with the community on Discord, the forum, or GitHub. Let's build the future, together. You know the drill!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>